# DATASCAPE 2026 — ConvNeXt-Base 384, A100 3-Jam, 5-Fold × 6 Epoch

Profil final yang disesuaikan untuk kredit A100 40 GB sekitar tiga jam:

- ConvNeXt-Base pretrained ImageNet-22k → ImageNet-1k, 88,6M parameter, input 384 px;
- tetap memakai 5 fold dari manifest agar ensemble dan estimasi OOF kuat;
- maksimal 6 epoch per fold, patience 2, warmup 1 epoch;
- physical batch mencoba 32 lebih dahulu dan effective batch tetap 128;
- A100 memakai BF16 tanpa gradient checkpointing;
- RGB dengan augmentasi konservatif, EMA, horizontal-flip TTA, dan kalibrasi cross-fitted;
- best checkpoint ditulis ke disk lokal selama training dan baru disalin sekali ke Drive per fold;
- pembuatan ZIP dimatikan untuk menghemat waktu; CSV, probabilitas, metrik, dan checkpoint tetap disimpan.

Aktifkan **A100 GPU**, lalu jalankan **Run all**.


In [ ]:
# Kaggle API bootstrap (configured for this project)
import os
from pathlib import Path

KAGGLE_API_TOKEN = 'KGAT_c2a604d54464b1bc7271fb95428085f2'
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN

kaggle_directory = Path.home() / '.kaggle'
kaggle_directory.mkdir(parents=True, exist_ok=True)
kaggle_access_token = kaggle_directory / 'access_token'
kaggle_access_token.write_text(KAGGLE_API_TOKEN + '\n')
kaggle_access_token.chmod(0o600)
print('Kaggle API token configured:', kaggle_access_token)


## AI Declaration dan batas pipeline

OpenAI Codex digunakan untuk riset model, coding, debugging, dan dokumentasi. Bobot `timm/convnext_base.fb_in22k_ft_in1k_384` diunduh dari Hugging Face Hub dan seluruh layer di-fine-tune pada data train resmi. Label student berasal dari teacher OOF yang hanya menggunakan data train resmi; tidak ada manual relabeling yang diterapkan ke training dan model eksternal tidak memberi label test.

Sumber model: <https://huggingface.co/timm/convnext_base.fb_in22k_ft_in1k_384>. Model card menyatakan lisensi Apache-2.0, 88,6 juta parameter, input 384×384, pretraining ImageNet-22k, dan fine-tuning ImageNet-1k.


In [ ]:
import importlib.util
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

required = {
    'numpy': 'numpy', 'pandas': 'pandas', 'PIL': 'Pillow',
    'matplotlib': 'matplotlib', 'seaborn': 'seaborn',
    'sklearn': 'scikit-learn', 'torch': 'torch',
    'torchvision': 'torchvision', 'tqdm': 'tqdm',
    'kornia': 'kornia', 'timm': 'timm', 'huggingface_hub': 'huggingface-hub',
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import kornia.augmentation as K
from kornia.color import hsv_to_rgb, lab_to_rgb, rgb_to_hsv, rgb_to_lab
from kornia.enhance import equalize_clahe
import matplotlib.pyplot as plt
import timm
from timm.utils import ModelEmaV3
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from IPython.display import display
from PIL import Image, ImageOps
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models
from tqdm.auto import tqdm

IN_COLAB = 'google.colab' in sys.modules
print('Colab:', IN_COLAB, '| PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())


## 1. Konfigurasi A100 40 GB / sekitar 3 jam

Input tetap 384 dan 5-fold tetap dipertahankan. Batas waktu dicapai dengan maksimal 6 epoch per fold. Physical batch diprobe mulai 32; jika lingkungan A100 tidak cukup karena overhead runtime, notebook fallback otomatis. BF16 dan channels-last dipakai untuk throughput A100.


In [ ]:
SEED = 42
CLASSES = ('Aman', 'Rusak', 'Lainnya')
CLASS_TO_INDEX = {label: index for index, label in enumerate(CLASSES)}
INDEX_TO_CLASS = {index: label for label, index in CLASS_TO_INDEX.items()}
COMPETITION = 'datascape-2026-data-mining-competition'

DATA_SOURCE = 'auto'
LOCAL_DATA_PATH = ''
PREPROCESSING_SOURCE = '/content/drive/MyDrive/DATASCAPE/relabel_convnext_tiny_320'
MANIFEST_FILENAME = 'student_training_manifest.csv'
UPLOAD_MANIFEST_IF_MISSING = True

USE_GOOGLE_DRIVE = True
OUTPUT_DIR = Path('/content/drive/MyDrive/DATASCAPE/final_convnext_base_student_384_amp_5fold_6epoch_a100')
LOCAL_OUTPUT_DIR = Path('artifacts/final_convnext_base_student_384_amp_5fold_6epoch_a100')
LOCAL_CHECKPOINT_ROOT = Path('/content/datascape_base_384_5fold_6epoch_checkpoints')
CREATE_OUTPUT_ZIP = False
EXTRACT_DIR = Path('/content/datascape_dataset')
PREPROCESS_EXTRACT_DIR = Path('/content/datascape_preprocessing_v1')

MODEL_ID = 'hf_hub:timm/convnext_base.fb_in22k_ft_in1k_384'
MODEL_HUB_URL = 'https://huggingface.co/timm/convnext_base.fb_in22k_ft_in1k_384'
MODEL_LICENSE = 'Apache-2.0'
DROP_RATE = 0.20
DROP_PATH_RATE = 0.20
MODEL_EMA_DECAY = 0.9997
AUTO_GRAD_CHECKPOINTING = True

IMAGE_SIZE = 384
TARGET_EFFECTIVE_BATCH_SIZE = 128
AUTO_BATCH_SIZE = True
BATCH_SIZE = 8
BATCH_CANDIDATES = (32, 24, 16, 12, 8, 6, 4, 2)
MAX_MEMORY_FRACTION = 0.90
MAX_EPOCHS = 6
PATIENCE = 2
WARMUP_EPOCHS = 1
REFERENCE_BATCH_SIZE = 128
REFERENCE_LR = 2e-4
MIN_LR = 1e-6
WEIGHT_DECAY = 5e-2
LABEL_SMOOTHING = 0.08
BALANCE_STRENGTH = 0.50
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 4
RESUME_FOLDS = True
FOLDS = 5
USE_HORIZONTAL_FLIP_TTA = True
CALIBRATE_CLASS_BIAS = True
CLASS_BIAS_LIMIT = 1.20
MIN_CLASS_BIAS_GAIN = 0.002

# Warna dipertahankan. Hanya 8% contoh augmented dibuat grayscale agar model
# tidak terlalu bergantung pada warna, tanpa membuang sinyal RGB seluruh dataset.
KEEP_RAW_PROBABILITY = 0.75
CLAHE_PROBABILITY = 0.08
HSV_PROBABILITY = 0.12
GRAYSCALE_PROBABILITY = 0.00
GAMMA_PROBABILITY = 0.08
BLUR_PROBABILITY = 0.03
JPEG_PROBABILITY = 0.05
DOWNSAMPLE_PROBABILITY = 0.03
ERASING_PROBABILITY = 0.02
CLAHE_CLIP_LIMIT = (1.2, 1.5)
HUE_SHIFT = 2
SATURATION_RANGE = (0.94, 1.06)
VALUE_RANGE = (0.94, 1.06)
GAMMA_RANGE = (0.92, 1.10)
JPEG_QUALITY_RANGE = (75.0, 98.0)
DOWNSAMPLE_SCALE_RANGE = (0.80, 0.95)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    OUTPUT_DIR = LOCAL_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision('high')
print('Output:', OUTPUT_DIR)


## 2. Hubungkan manifest relabel dan dataset resmi

Notebook membaca `student_training_manifest.csv` dari output relabeling. Kolom `label` adalah target training otomatis final. Kolom `label_original` dipertahankan sebagai target validation untuk diagnostik yang lebih konservatif.

Notebook lebih dahulu mencoba path Drive yang dikonfigurasi, lalu mencari manifest di Drive akun aktif dan `/content`. Jika akun Colab berbeda dan file belum tersedia, upload picker akan muncul otomatis. Ambil hanya `student_training_manifest.csv` dari akun lama; checkpoint teacher tidak diperlukan untuk final training.

Metrik student OOF tetap bukan pengganti nested evaluation penuh, karena pseudo-label global dapat membawa informasi lintas fold secara tidak langsung. Metrik dipakai sebagai diagnostik dan pemilihan checkpoint; submission tetap berasal dari ensemble lima model.


In [ ]:
def get_kaggle_token():
    token = os.environ.get('KAGGLE_API_TOKEN', '').strip()
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = (userdata.get('KAGGLE_API_TOKEN') or token).strip()
        except Exception:
            pass
    if not token:
        raise RuntimeError('Set KAGGLE_API_TOKEN di Colab Secrets atau environment.')
    return token


def safe_extract(zip_path, destination):
    zip_path, destination = Path(zip_path), Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(root):
                raise ValueError(f'Unsafe ZIP member: {member.filename}')
        archive.extractall(destination)
    return destination


def find_file_root(start, filename):
    start = Path(start).expanduser()
    if start.is_file() and start.name == filename:
        return start.parent
    if start.is_dir() and (start / filename).is_file():
        return start
    if start.is_dir():
        matches = list(start.rglob(filename))
        if matches:
            return matches[0].parent
    raise FileNotFoundError(f'{filename} tidak ditemukan di {start}')


def resolve_preprocessing_root():
    # Path akun lama tetap menjadi pilihan pertama, tetapi tidak lagi menghentikan
    # notebook saat Colab dijalankan memakai akun Google yang berbeda.
    if PREPROCESSING_SOURCE:
        source = Path(PREPROCESSING_SOURCE).expanduser()
        if source.exists():
            if source.suffix.lower() == '.zip':
                extracted = safe_extract(source, PREPROCESS_EXTRACT_DIR)
                return find_file_root(extracted, MANIFEST_FILENAME)
            return find_file_root(source, MANIFEST_FILENAME)
        print('Path manifest terkonfigurasi tidak ada pada akun ini:', source)

    candidates = [
        Path('/content'),
        Path('/content/datascape_preprocessing'),
        Path('/content/artifacts/preprocessing_colab'),
        Path.cwd(),
        Path.cwd() / 'artifacts' / 'preprocessing_colab',
        Path.cwd() / 'artifacts' / 'preprocessing_v1',
    ]
    for candidate in candidates:
        if (candidate / MANIFEST_FILENAME).is_file():
            return candidate

    if IN_COLAB:
        drive_roots = [
            Path('/content/drive/MyDrive/DATASCAPE'),
            Path('/content/drive/MyDrive'),
        ]
        for drive_root in drive_roots:
            if not drive_root.is_dir():
                continue
            direct = drive_root / MANIFEST_FILENAME
            if direct.is_file():
                return direct.parent
            matches = sorted(drive_root.glob(f'*/{MANIFEST_FILENAME}'))
            if matches:
                print('Manifest ditemukan di Drive akun aktif:', matches[0])
                return matches[0].parent

        archives = sorted(Path('/content').glob('*relabel*.zip'))
        archives += sorted(Path('/content').glob('*preprocess*.zip'))
        for archive in archives:
            try:
                return find_file_root(
                    safe_extract(archive, PREPROCESS_EXTRACT_DIR), MANIFEST_FILENAME
                )
            except FileNotFoundError:
                pass

        if UPLOAD_MANIFEST_IF_MISSING:
            from google.colab import files
            print(
                '\nManifest belum ada di Drive akun aktif. '
                'Pilih student_training_manifest.csv dari akun/run labeling sebelumnya.'
            )
            uploaded = files.upload()
            uploaded_path = Path('/content') / MANIFEST_FILENAME
            if MANIFEST_FILENAME in uploaded:
                uploaded_path.write_bytes(uploaded[MANIFEST_FILENAME])
                print('Manifest di-upload ke:', uploaded_path)
                return uploaded_path.parent
            csv_uploads = [name for name in uploaded if name.lower().endswith('.csv')]
            if len(csv_uploads) == 1:
                uploaded_path.write_bytes(uploaded[csv_uploads[0]])
                print(f'Manifest {csv_uploads[0]} disimpan sebagai:', uploaded_path)
                return uploaded_path.parent
            raise FileNotFoundError(
                f'Upload harus berisi tepat satu file CSV; diterima: {sorted(uploaded)}'
            )

    raise FileNotFoundError(
        f'{MANIFEST_FILENAME} tidak ditemukan. Salin file dari output relabeling '
        'akun lama ke /content atau isi PREPROCESSING_SOURCE.'
    )


def find_dataset_root(start):
    start = Path(start).expanduser().resolve()
    candidates = [start]
    if start.is_dir():
        candidates.extend(path.parent for path in start.rglob('sample_submission.csv'))
    for root in candidates:
        if (root / 'sample_submission.csv').is_file() and (root / 'Test_Set').is_dir():
            if all((root / 'Train_Set' / label).is_dir() for label in CLASSES):
                return root
    raise FileNotFoundError(f'Root dataset tidak ditemukan di {start}')


def resolve_dataset_root():
    if DATA_SOURCE in {'auto', 'folder'} and LOCAL_DATA_PATH:
        local = Path(LOCAL_DATA_PATH).expanduser()
        if local.exists():
            return find_dataset_root(local)
        if DATA_SOURCE == 'folder':
            raise FileNotFoundError(local)
    for candidate in [Path.cwd() / COMPETITION, Path('/content') / COMPETITION]:
        if DATA_SOURCE == 'auto' and candidate.exists():
            return find_dataset_root(candidate)
    if importlib.util.find_spec('kagglehub') is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'])
    import kagglehub
    from kagglehub.config import set_kaggle_api_token
    set_kaggle_api_token(get_kaggle_token())
    downloaded = Path(kagglehub.competition_download(COMPETITION))
    if downloaded.is_file() and downloaded.suffix.lower() == '.zip':
        return find_dataset_root(safe_extract(downloaded, EXTRACT_DIR))
    return find_dataset_root(downloaded)


PREPROCESS_ROOT = resolve_preprocessing_root()
DATA_ROOT = resolve_dataset_root()
print('Preprocessing V1 root:', PREPROCESS_ROOT)
print('Dataset root:', DATA_ROOT)


In [ ]:
train_frame = pd.read_csv(PREPROCESS_ROOT / MANIFEST_FILENAME)
required = {
    'filepath', 'label', 'label_original', 'label_final', 'label_source',
    'label_decision', 'fold', 'split_group', 'pixel_sha256',
}
missing = required - set(train_frame.columns)
if missing:
    raise ValueError(f'Kolom student manifest tidak lengkap: {sorted(missing)}')

train_frame = train_frame.reset_index(drop=True)
if not train_frame.label.eq(train_frame.label_final).all():
    raise ValueError('Kolom label harus identik dengan label_final pada student manifest.')
if not set(train_frame.label).issubset(CLASSES) or set(train_frame.label_original) != set(CLASSES):
    raise ValueError('Kelas student manifest tidak sesuai.')

train_frame['image_path'] = train_frame.filepath.map(lambda value: str(DATA_ROOT / Path(value)))
missing_paths = train_frame.loc[~train_frame.image_path.map(lambda value: Path(value).is_file()), 'filepath']
if len(missing_paths):
    raise FileNotFoundError(f'{len(missing_paths)} gambar train tidak ditemukan; contoh: {missing_paths.iloc[0]}')
if train_frame.pixel_sha256.duplicated().any():
    raise ValueError('Exact duplicate masih ada di student manifest.')
if train_frame.groupby('split_group').fold.nunique().max() != 1:
    raise ValueError('Grup near-duplicate menyeberangi fold.')

template = pd.read_csv(DATA_ROOT / 'sample_submission.csv', dtype={'filename': str})
test_root = DATA_ROOT / 'Test_Set'
by_name = {path.name: path for path in test_root.rglob('*') if path.is_file()}
missing_test = [name for name in template.filename.astype(str) if name not in by_name]
if missing_test:
    raise FileNotFoundError(f'{len(missing_test)} file test tidak ditemukan; contoh: {missing_test[0]}')
test_frame = pd.DataFrame({
    'filename': template.filename.astype(str),
    'image_path': [str(by_name[name]) for name in template.filename.astype(str)],
})

print('Train student:', len(train_frame), '| Test:', len(test_frame))
print('Perubahan label otomatis:', int(train_frame.label.ne(train_frame.label_original).sum()))
display(pd.crosstab(train_frame.fold, train_frame.label).reindex(columns=CLASSES, fill_value=0))
display(pd.crosstab(train_frame.label_original, train_frame.label).reindex(index=CLASSES, columns=CLASSES, fill_value=0))


## 3. Raw-image dataset dan augmentasi RGB moderat

CPU hanya melakukan decode, koreksi orientasi EXIF, konversi RGB, letterbox, dan transfer tensor uint8. Training tetap memakai RGB karena bobot pretrained belajar fitur tiga kanal dan warna dapat membantu membedakan permukaan serta pencahayaan. Grayscale sintetis dimatikan karena variasi tersebut sudah tersedia secara alami pada dataset.

Transform GPU dibuat moderat: horizontal flip, affine kecil, luminance/warna yang sangat ringan serta blur/JPEG/downsample/erasing berprobabilitas sangat rendah. Tidak ada vertical flip atau crop agresif yang dapat membuang area kerusakan. Validation dan test selalu memakai RGB asli, letterbox, dan normalisasi ImageNet.


In [ ]:
class Letterbox:
    def __init__(self, size):
        self.size = int(size)
        self.fill = tuple(int(round(value * 255)) for value in IMAGENET_MEAN)
    def __call__(self, image):
        return ImageOps.pad(image, (self.size, self.size), Image.Resampling.BILINEAR,
                            color=self.fill, centering=(0.5, 0.5))


def load_rgb_image(path):
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image)
        if image.mode == 'P':
            image = image.convert('RGBA')
        if image.mode in {'RGBA', 'LA'} or 'transparency' in image.info:
            rgba = image.convert('RGBA')
            background = Image.new('RGBA', rgba.size, tuple(int(v * 255) for v in IMAGENET_MEAN) + (255,))
            image = Image.alpha_composite(background, rgba).convert('RGB')
        else:
            image = image.convert('RGB')
        return image.copy()


class ToUint8Tensor:
    def __call__(self, image):
        array = np.asarray(image, dtype=np.uint8).copy()
        return torch.from_numpy(array).permute(2, 0, 1)


CPU_LETTERBOX = Letterbox(IMAGE_SIZE)
CPU_TO_TENSOR = ToUint8Tensor()


def cpu_transform(image):
    return CPU_TO_TENSOR(CPU_LETTERBOX(image))


class PackageDataset(Dataset):
    def __init__(self, frame, training=False, with_labels=True, target_column='label'):
        self.frame = frame.reset_index(drop=True)
        self.training = training
        self.with_labels = with_labels
        self.target_column = target_column
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        tensor = cpu_transform(load_rgb_image(row.image_path))
        if self.with_labels:
            return tensor, CLASS_TO_INDEX[row[self.target_column]], index
        return tensor, index


def balanced_sampler(frame):
    counts = frame.label.value_counts()
    weights = frame.label.map(lambda label: counts[label] ** -BALANCE_STRENGTH).to_numpy(copy=True)
    return WeightedRandomSampler(torch.tensor(weights, dtype=torch.double), len(frame), replacement=True)


def make_loader(dataset, batch_size, sampler=None):
    options = dict(dataset=dataset, batch_size=batch_size, sampler=sampler, shuffle=False,
                   num_workers=NUM_WORKERS, pin_memory=True)
    if NUM_WORKERS > 0:
        options.update(persistent_workers=True, prefetch_factor=2)
    return DataLoader(**options)

preview = PackageDataset(train_frame.head(4), training=True, target_column='label')
print('Preview shape/dtype:', tuple(preview[0][0].shape), preview[0][0].dtype)


## 4. Full fine-tuning Base dengan AMP, EMA, dan gradient checkpointing

Seluruh layer ConvNeXt-Base di-fine-tune. L4/A100 memakai BF16; T4 memakai FP16 dengan `GradScaler`. Pada T4/L4, gradient checkpointing menukar sebagian kecepatan dengan penggunaan VRAM yang lebih rendah. Gradient accumulation menjaga effective batch sekitar 128, EMA dipakai untuk validation/inference, dan checkpoint dipilih berdasarkan validation macro-F1 terhadap label asli.


In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda':
    raise RuntimeError('Aktifkan GPU Colab.')

GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
GPU_COMPUTE_CAPABILITY = torch.cuda.get_device_capability(0)
SUPPORTS_NATIVE_BF16 = (
    GPU_COMPUTE_CAPABILITY[0] >= 8 and torch.cuda.is_bf16_supported()
)
AMP_DTYPE = torch.bfloat16 if SUPPORTS_NATIVE_BF16 else torch.float16
AMP_NAME = str(AMP_DTYPE).replace('torch.', '')
USE_GRAD_SCALER = AMP_DTYPE == torch.float16
USE_GRAD_CHECKPOINTING = AUTO_GRAD_CHECKPOINTING and GPU_VRAM_GIB <= 28
print(
    'GPU:', GPU_NAME, '| VRAM:', f'{GPU_VRAM_GIB:.1f} GiB',
    '| AMP:', AMP_NAME, '| gradient checkpointing:', USE_GRAD_CHECKPOINTING,
)

GPU_SPATIAL = K.AugmentationSequential(
    K.RandomHorizontalFlip(p=0.5),
    K.RandomAffine(
        degrees=3.0, translate=(0.02, 0.02), scale=(0.97, 1.03),
        padding_mode='border', p=0.30,
    ),
    data_keys=['input'],
).to(DEVICE)

GPU_PHOTOMETRIC = K.AugmentationSequential(
    K.RandomGamma(gamma=GAMMA_RANGE, gain=(1.0, 1.0), p=GAMMA_PROBABILITY),
    K.RandomGrayscale(p=GRAYSCALE_PROBABILITY),
    K.RandomGaussianBlur(kernel_size=(5, 5), sigma=(0.3, 0.7), p=BLUR_PROBABILITY),
    K.RandomJPEG(jpeg_quality=JPEG_QUALITY_RANGE, p=JPEG_PROBABILITY),
    data_keys=['input'],
).to(DEVICE)

NORMALIZE_MEAN = torch.tensor(IMAGENET_MEAN, device=DEVICE).view(1, 3, 1, 1)
NORMALIZE_STD = torch.tensor(IMAGENET_STD, device=DEVICE).view(1, 3, 1, 1)


def to_gpu_float(images):
    return images.to(
        DEVICE, dtype=torch.float32, memory_format=torch.channels_last, non_blocking=True
    ).div_(255.0)


def normalize_batch(images):
    return ((images - NORMALIZE_MEAN) / NORMALIZE_STD).contiguous(memory_format=torch.channels_last)


def random_lab_clahe(images):
    apply_mask = torch.rand(images.shape[0], device=DEVICE) < CLAHE_PROBABILITY
    if not apply_mask.any():
        return images
    selected = images[apply_mask]
    lab = rgb_to_lab(selected)
    luminance = (lab[:, :1] / 100.0).clamp(0.0, 1.0)
    clip_limit = float(torch.empty((), device=DEVICE).uniform_(*CLAHE_CLIP_LIMIT))
    corrected = equalize_clahe(luminance, clip_limit=clip_limit, grid_size=(8, 8))
    corrected_lab = torch.cat((corrected * 100.0, lab[:, 1:]), dim=1)
    result = images.clone()
    result[apply_mask] = lab_to_rgb(corrected_lab).clamp(0.0, 1.0)
    return result


def random_hsv(images):
    apply_mask = torch.rand(images.shape[0], 1, 1, 1, device=DEVICE) < HSV_PROBABILITY
    hsv = rgb_to_hsv(images)
    hue = torch.empty(images.shape[0], 1, 1, 1, device=DEVICE).uniform_(
        -HUE_SHIFT * math.pi / 180.0, HUE_SHIFT * math.pi / 180.0
    )
    saturation = torch.empty(images.shape[0], 1, 1, 1, device=DEVICE).uniform_(*SATURATION_RANGE)
    value = torch.empty(images.shape[0], 1, 1, 1, device=DEVICE).uniform_(*VALUE_RANGE)
    adjusted = torch.cat((
        torch.remainder(hsv[:, :1] + hue, 2.0 * math.pi),
        (hsv[:, 1:2] * saturation).clamp(0.0, 1.0),
        (hsv[:, 2:3] * value).clamp(0.0, 1.0),
    ), dim=1)
    return torch.where(apply_mask, hsv_to_rgb(adjusted), images)


def random_downsample(images):
    scale = float(torch.empty((), device=DEVICE).uniform_(*DOWNSAMPLE_SCALE_RANGE))
    height = max(32, int(images.shape[-2] * scale))
    width = max(32, int(images.shape[-1] * scale))
    degraded = F.interpolate(images, size=(height, width), mode='bilinear', align_corners=False)
    degraded = F.interpolate(degraded, size=images.shape[-2:], mode='bilinear', align_corners=False)
    mask = torch.rand(images.shape[0], 1, 1, 1, device=DEVICE) < DOWNSAMPLE_PROBABILITY
    return torch.where(mask, degraded, images)


def random_erasing(images):
    result = images.clone()
    height, width = images.shape[-2:]
    apply_indices = torch.where(
        torch.rand(images.shape[0], device=DEVICE) < ERASING_PROBABILITY
    )[0]
    for index_tensor in apply_indices:
        index = int(index_tensor.item())
        area = float(height * width)
        target_area = area * float(torch.empty((), device=DEVICE).uniform_(0.02, 0.06))
        aspect = math.exp(float(torch.empty((), device=DEVICE).uniform_(math.log(0.4), math.log(2.5))))
        erase_height = min(height, max(1, int(round(math.sqrt(target_area * aspect)))))
        erase_width = min(width, max(1, int(round(math.sqrt(target_area / aspect)))))
        top = int(torch.randint(0, height - erase_height + 1, (), device=DEVICE))
        left = int(torch.randint(0, width - erase_width + 1, (), device=DEVICE))
        fill = result[index].mean(dim=(1, 2), keepdim=True)
        result[index, :, top:top + erase_height, left:left + erase_width] = fill
    return result


def prepare_train_batch(images):
    spatial = GPU_SPATIAL(to_gpu_float(images))
    use_augmented = torch.rand(spatial.shape[0], device=DEVICE) >= KEEP_RAW_PROBABILITY
    images = spatial.clone()
    if use_augmented.any():
        augmented = random_lab_clahe(spatial[use_augmented])
        augmented = random_hsv(augmented)
        augmented = GPU_PHOTOMETRIC(augmented)
        images[use_augmented] = random_downsample(augmented)
    return normalize_batch(random_erasing(images))


def prepare_eval_batch(images):
    return normalize_batch(to_gpu_float(images))


def build_model():
    model = timm.create_model(
        MODEL_ID, pretrained=True, num_classes=len(CLASSES),
        drop_rate=DROP_RATE, drop_path_rate=DROP_PATH_RATE,
    )
    if USE_GRAD_CHECKPOINTING:
        if not hasattr(model, 'set_grad_checkpointing'):
            raise RuntimeError('Model timm ini tidak mendukung gradient checkpointing.')
        model.set_grad_checkpointing(True)
    return model.to(DEVICE, memory_format=torch.channels_last)


def candidate_batches_for_gpu():
    if GPU_VRAM_GIB <= 18:
        maximum = 8       # Tesla T4 16 GB
    elif GPU_VRAM_GIB <= 28:
        maximum = 16      # L4 24 GB
    elif GPU_VRAM_GIB <= 52:
        maximum = 32      # A100 40 GB
    else:
        maximum = 64      # A100 80 GB / H100
    candidates = tuple(batch for batch in BATCH_CANDIDATES if batch <= maximum)
    print(f'VRAM: {GPU_VRAM_GIB:.1f} GiB | Base batch probe: {candidates}')
    return candidates


def probe_batch():
    if not AUTO_BATCH_SIZE:
        return BATCH_SIZE
    total = torch.cuda.get_device_properties(0).total_memory
    for batch in candidate_batches_for_gpu():
        model = ema = optimizer = scaler = images = prepared = targets = loss = None
        try:
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            model = build_model()
            ema = ModelEmaV3(model, decay=MODEL_EMA_DECAY)
            optimizer = torch.optim.AdamW(model.parameters(), lr=REFERENCE_LR, weight_decay=WEIGHT_DECAY)
            scaler = torch.amp.GradScaler('cuda', enabled=USE_GRAD_SCALER)
            images = torch.randint(
                0, 256, (batch, 3, IMAGE_SIZE, IMAGE_SIZE),
                dtype=torch.uint8, device=DEVICE,
            )
            prepared = prepare_train_batch(images)
            targets = torch.randint(0, len(CLASSES), (batch,), device=DEVICE)
            with torch.autocast('cuda', dtype=AMP_DTYPE):
                loss = nn.functional.cross_entropy(model(prepared), targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            ema.update(model)
            torch.cuda.synchronize()
            fraction = torch.cuda.max_memory_reserved() / total
            print(f'batch {batch}: {fraction:.1%} VRAM')
            if fraction <= MAX_MEMORY_FRACTION:
                return batch
        except (torch.cuda.OutOfMemoryError, RuntimeError) as error:
            if not isinstance(error, torch.cuda.OutOfMemoryError) and 'out of memory' not in str(error).lower():
                raise
            print(f'batch {batch}: OOM')
        finally:
            del model, ema, optimizer, scaler, images, prepared, targets, loss
            torch.cuda.empty_cache()
    raise RuntimeError('Tidak ada batch size yang muat, termasuk batch 4.')


BATCH_SIZE = probe_batch()
GRAD_ACCUM_STEPS = max(1, math.ceil(TARGET_EFFECTIVE_BATCH_SIZE / BATCH_SIZE))
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRAD_ACCUM_STEPS
EVAL_BATCH_SIZE = min(128, max(8, BATCH_SIZE * 2))
BASE_LR = REFERENCE_LR * math.sqrt(EFFECTIVE_BATCH_SIZE / REFERENCE_BATCH_SIZE)
print(
    'Physical batch:', BATCH_SIZE,
    '| accumulation:', GRAD_ACCUM_STEPS,
    '| effective batch:', EFFECTIVE_BATCH_SIZE,
    '| eval batch:', EVAL_BATCH_SIZE,
    '| base LR:', BASE_LR,
)


def learning_rate_at(epoch):
    if epoch < WARMUP_EPOCHS:
        return BASE_LR * (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(MAX_EPOCHS - WARMUP_EPOCHS - 1, 1)
    return MIN_LR + (BASE_LR - MIN_LR) * 0.5 * (1 + math.cos(math.pi * min(progress, 1)))


def predict(model, data_loader, labeled, use_tta=False):
    model.eval()
    probabilities, targets_all, indices_all = [], [], []
    with torch.inference_mode():
        for batch in tqdm(data_loader, desc='predict', leave=False):
            if labeled:
                images, targets, indices = batch
                targets_all.append(targets.numpy())
            else:
                images, indices = batch
            images = prepare_eval_batch(images)
            with torch.autocast('cuda', dtype=AMP_DTYPE):
                probs = model(images).softmax(1)
                if use_tta:
                    probs = 0.5 * (probs + model(torch.flip(images, dims=[3])).softmax(1))
            probabilities.append(probs.float().cpu().numpy())
            indices_all.append(indices.numpy())
    order = np.argsort(np.concatenate(indices_all))
    result = {'probabilities': np.concatenate(probabilities)[order]}
    if labeled:
        result['targets'] = np.concatenate(targets_all)[order]
    return result


def train_fold(fold):
    output_fold_dir = OUTPUT_DIR / f'fold_{fold}'
    output_fold_dir.mkdir(parents=True, exist_ok=True)
    local_fold_dir = LOCAL_CHECKPOINT_ROOT / f'fold_{fold}'
    local_fold_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = local_fold_dir / 'best.pt'
    history_path = local_fold_dir / 'history.csv'
    output_checkpoint_path = output_fold_dir / 'best.pt'
    complete_path = output_fold_dir / 'complete.json'
    train = train_frame[train_frame.fold.ne(fold)].reset_index(drop=True)
    valid = train_frame[train_frame.fold.eq(fold)].reset_index(drop=True)
    if RESUME_FOLDS and output_checkpoint_path.is_file() and complete_path.is_file():
        checkpoint = torch.load(output_checkpoint_path, map_location=DEVICE, weights_only=False)
        model = build_model()
        model.load_state_dict(checkpoint['model_state'])
        return model, valid, checkpoint

    train_loader = make_loader(PackageDataset(train, training=True), BATCH_SIZE, balanced_sampler(train))
    valid_loader = make_loader(PackageDataset(valid, target_column='label_original'), EVAL_BATCH_SIZE)
    model = build_model()
    ema = ModelEmaV3(model, decay=MODEL_EMA_DECAY)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_GRAD_SCALER)
    best, stale, history = -1.0, 0, []

    for epoch in range(MAX_EPOCHS):
        lr = learning_rate_at(epoch)
        for group in optimizer.param_groups:
            group['lr'] = lr
        model.train()
        optimizer.zero_grad(set_to_none=True)
        total_loss, total, started = 0.0, 0, time.time()
        number_of_batches = len(train_loader)
        progress = tqdm(train_loader, desc=f'fold {fold} epoch {epoch + 1}', leave=False)
        for step, (images, targets, _) in enumerate(progress):
            images = prepare_train_batch(images)
            targets = targets.to(DEVICE, non_blocking=True)
            window_start = (step // GRAD_ACCUM_STEPS) * GRAD_ACCUM_STEPS
            window_size = min(GRAD_ACCUM_STEPS, number_of_batches - window_start)
            with torch.autocast('cuda', dtype=AMP_DTYPE):
                raw_loss = criterion(model(images), targets)
                scaled_loss = raw_loss / window_size
            scaler.scale(scaled_loss).backward()
            should_step = (step + 1) % GRAD_ACCUM_STEPS == 0 or step + 1 == number_of_batches
            if should_step:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                ema.update(model)
            total_loss += float(raw_loss.item()) * len(images)
            total += len(images)

        result = predict(ema.module, valid_loader, True)
        score = f1_score(
            result['targets'], result['probabilities'].argmax(1),
            average='macro', zero_division=0,
        )
        row = {
            'epoch': epoch + 1, 'lr': lr, 'train_loss': total_loss / total,
            'valid_macro_f1': score, 'seconds': time.time() - started,
            'physical_batch_size': BATCH_SIZE,
            'gradient_accumulation_steps': GRAD_ACCUM_STEPS,
            'effective_batch_size': EFFECTIVE_BATCH_SIZE,
            'amp_dtype': AMP_NAME,
        }
        history.append(row)
        pd.DataFrame(history).to_csv(history_path, index=False)
        print(
            f"fold {fold} epoch {epoch + 1}: "
            f"loss={row['train_loss']:.4f} macro_f1={score:.4f} "
            f"time={row['seconds'] / 60:.1f}m"
        )
        if score > best + 1e-6:
            best, stale = score, 0
            torch.save({
                'model_state': ema.module.state_dict(),
                'epoch': epoch + 1,
                'macro_f1': score,
                'amp_dtype': AMP_NAME,
                'physical_batch_size': BATCH_SIZE,
                'gradient_accumulation_steps': GRAD_ACCUM_STEPS,
                'effective_batch_size': EFFECTIVE_BATCH_SIZE,
            }, checkpoint_path)
        else:
            stale += 1
            if stale >= PATIENCE:
                break

    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint['model_state'])
    # Satu kali transfer checkpoint besar ke Drive setelah fold selesai.
    shutil.copy2(checkpoint_path, output_checkpoint_path)
    shutil.copy2(history_path, output_fold_dir / 'history.csv')
    complete_path.write_text(json.dumps({
        'fold': fold,
        'epoch': checkpoint['epoch'],
        'macro_f1': checkpoint['macro_f1'],
        'physical_batch_size': BATCH_SIZE,
        'gradient_accumulation_steps': GRAD_ACCUM_STEPS,
        'effective_batch_size': EFFECTIVE_BATCH_SIZE,
        'amp_dtype': AMP_NAME,
        'profile': 'A100 40GB, 5-fold x maximum 6 epochs',
    }, indent=2))
    del ema
    return model, valid, checkpoint


## 5. Ensemble test dan submission

Setiap fold menghasilkan probabilitas validation dan test. Training memakai label student, sedangkan target validation tetap label asli. Submission memakai rata-rata probabilitas lima model dan horizontal-flip TTA. Kalibrasi bias kelas memakai target asli, tetapi hasil OOF harus dibaca sebagai diagnostik karena relabeling global bukan nested CV penuh.


In [ ]:
oof_probabilities = np.full((len(train_frame), len(CLASSES)), np.nan, dtype=np.float32)
test_probability_sum = np.zeros((len(test_frame), len(CLASSES)), dtype=np.float64)
test_loader = make_loader(PackageDataset(test_frame, with_labels=False), EVAL_BATCH_SIZE)
fold_rows = []

for fold in range(FOLDS):
    model, valid, checkpoint = train_fold(fold)
    valid_loader = make_loader(PackageDataset(valid, target_column='label_original'), EVAL_BATCH_SIZE)
    valid_result = predict(model, valid_loader, True, use_tta=USE_HORIZONTAL_FLIP_TTA)
    global_indices = train_frame.index[train_frame.fold.eq(fold)].to_numpy()
    oof_probabilities[global_indices] = valid_result['probabilities']

    test_result = predict(model, test_loader, False, use_tta=USE_HORIZONTAL_FLIP_TTA)
    test_probability_sum += test_result['probabilities']
    np.save(OUTPUT_DIR / f'fold_{fold}' / 'test_probabilities.npy', test_result['probabilities'])
    fold_rows.append({'fold': fold, 'best_epoch': checkpoint['epoch'], 'macro_f1': checkpoint['macro_f1']})
    del model
    torch.cuda.empty_cache()

assert np.isfinite(oof_probabilities).all()
test_probabilities = (test_probability_sum / FOLDS).astype(np.float32)
np.save(OUTPUT_DIR / 'oof_probabilities.npy', oof_probabilities)
pd.DataFrame(fold_rows).to_csv(OUTPUT_DIR / 'fold_metrics.csv', index=False)

def apply_class_bias(probabilities, bias):
    scores = np.log(np.clip(probabilities, 1e-8, 1.0)) + np.asarray(bias, dtype=np.float64)
    scores -= scores.max(axis=1, keepdims=True)
    calibrated = np.exp(scores)
    return (calibrated / calibrated.sum(axis=1, keepdims=True)).astype(np.float32)


def optimize_class_bias(probabilities, targets):
    best_bias = np.zeros(len(CLASSES), dtype=np.float64)
    best_score = f1_score(targets, probabilities.argmax(1), average='macro', zero_division=0)
    # Aman menjadi reference (bias=0); hanya dua derajat bebas agar tidak overfit.
    for radius, step in [(CLASS_BIAS_LIMIT, 0.10), (0.12, 0.02)]:
        center = best_bias.copy()
        rusak_values = np.arange(center[1] - radius, center[1] + radius + step / 2, step)
        lainnya_values = np.arange(center[2] - radius, center[2] + radius + step / 2, step)
        for rusak_bias in rusak_values:
            for lainnya_bias in lainnya_values:
                candidate = np.array([0.0, rusak_bias, lainnya_bias])
                predicted = (np.log(np.clip(probabilities, 1e-8, 1.0)) + candidate).argmax(1)
                score = f1_score(targets, predicted, average='macro', zero_division=0)
                if score > best_score + 1e-12 or (
                    abs(score - best_score) <= 1e-12
                    and np.linalg.norm(candidate) < np.linalg.norm(best_bias)
                ):
                    best_score, best_bias = score, candidate
    return best_bias, best_score


targets = train_frame.label_original.map(CLASS_TO_INDEX).to_numpy()
raw_predictions = oof_probabilities.argmax(1)
raw_macro_f1 = f1_score(targets, raw_predictions, average='macro', zero_division=0)

# Setiap fold dikalibrasi memakai OOF dari empat fold lain. Ini memberi estimasi
# kalibrasi yang jujur dan lima bias independen untuk diringkas secara robust.
calibrated_oof = np.zeros_like(oof_probabilities)
fold_biases = []
calibration_rows = []
for held_out_fold in range(FOLDS):
    calibration_mask = train_frame.fold.ne(held_out_fold).to_numpy()
    held_out_mask = ~calibration_mask
    bias, calibration_score = optimize_class_bias(
        oof_probabilities[calibration_mask], targets[calibration_mask]
    )
    calibrated_oof[held_out_mask] = apply_class_bias(oof_probabilities[held_out_mask], bias)
    held_out_score = f1_score(
        targets[held_out_mask], calibrated_oof[held_out_mask].argmax(1),
        average='macro', zero_division=0,
    )
    fold_biases.append(bias)
    calibration_rows.append({
        'held_out_fold': held_out_fold,
        'calibration_macro_f1': calibration_score,
        'held_out_macro_f1': held_out_score,
        **{f'bias_{label}': float(bias[index]) for index, label in enumerate(CLASSES)},
    })

crossfit_calibrated_predictions = calibrated_oof.argmax(1)
crossfit_calibrated_macro_f1 = f1_score(
    targets, crossfit_calibrated_predictions, average='macro', zero_division=0
)
calibration_accepted = (
    CALIBRATE_CLASS_BIAS
    and crossfit_calibrated_macro_f1 >= raw_macro_f1 + MIN_CLASS_BIAS_GAIN
)
final_class_bias = (
    np.median(np.stack(fold_biases), axis=0)
    if calibration_accepted else np.zeros(len(CLASSES))
)
calibrated_test_probabilities = apply_class_bias(test_probabilities, final_class_bias)
predictions = crossfit_calibrated_predictions if calibration_accepted else raw_predictions
macro_f1 = crossfit_calibrated_macro_f1 if calibration_accepted else raw_macro_f1

np.save(OUTPUT_DIR / 'oof_probabilities_calibrated.npy', calibrated_oof)
np.save(OUTPUT_DIR / 'test_probabilities_raw.npy', test_probabilities)
np.save(OUTPUT_DIR / 'test_probabilities.npy', calibrated_test_probabilities)
pd.DataFrame(calibration_rows).to_csv(OUTPUT_DIR / 'class_bias_crossfit.csv', index=False)

report = pd.DataFrame(classification_report(targets, predictions, target_names=CLASSES,
                                            output_dict=True, zero_division=0)).T
report.to_csv(OUTPUT_DIR / 'oof_classification_report.csv')
display(pd.DataFrame(calibration_rows))
display(report)
print(f'Raw OOF macro-F1: {raw_macro_f1:.4f} | Cross-fitted calibrated: {crossfit_calibrated_macro_f1:.4f}')
print('Calibration accepted:', calibration_accepted, '| Final class bias:', dict(zip(CLASSES, final_class_bias.round(4))))

matrix = confusion_matrix(targets, predictions, normalize='true')
plt.figure(figsize=(6, 5))
sns.heatmap(matrix, annot=True, fmt='.3f', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.title(f'OOF macro F1 = {macro_f1:.4f}')
plt.xlabel('Prediksi'); plt.ylabel('Label'); plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'oof_confusion_matrix.png', dpi=170); plt.show()

submission = template.copy()
submission['label'] = [INDEX_TO_CLASS[index] for index in calibrated_test_probabilities.argmax(1)]
submission.to_csv(OUTPUT_DIR / 'submission_convnext_base_student_384_amp_5fold_6epoch_a100.csv', index=False)
probability_frame = pd.DataFrame(calibrated_test_probabilities, columns=[f'prob_{label}' for label in CLASSES])
probability_frame.insert(0, 'filename', template.filename)
probability_frame.to_csv(OUTPUT_DIR / 'test_probabilities.csv', index=False)

summary = {
    'model': 'ConvNeXt-Base 384 pretrained from Hugging Face Hub, fully fine-tuned',
    'model_id': MODEL_ID,
    'model_hub_url': MODEL_HUB_URL,
    'model_license': MODEL_LICENSE,
    'pretraining': 'ImageNet-22k then ImageNet-1k',
    'parameters_millions': 88.6,
    'training_images': int(len(train_frame)),
    'label_source': 'student_training_manifest.csv (ConvNeXt OOF automatic labels)',
    'relabeling_used': True,
    'automatic_relabels': int(train_frame.label.ne(train_frame.label_original).sum()),
    'validation_target': 'label_original',
    'amp_enabled': True,
    'amp_dtype': AMP_NAME,
    'oof_metric_caveat': 'diagnostic only; pseudo labels were not generated with nested outer-fold isolation',
    'oof_macro_f1_raw': float(raw_macro_f1),
    'oof_macro_f1_crossfit_calibrated': float(crossfit_calibrated_macro_f1),
    'oof_macro_f1_selected': float(macro_f1),
    'class_bias_calibration_requested': CALIBRATE_CLASS_BIAS,
    'class_bias_calibration_applied': bool(calibration_accepted),
    'final_class_bias': {label: float(final_class_bias[index]) for index, label in enumerate(CLASSES)},
    'folds': FOLDS,
    'max_epochs': MAX_EPOCHS,
    'patience': PATIENCE,
    'training_profile': 'A100 40GB, 5-fold x maximum 6 epochs',
    'zip_created': CREATE_OUTPUT_ZIP,
    'image_size': IMAGE_SIZE,
    'physical_batch_size': BATCH_SIZE,
    'gradient_accumulation_steps': GRAD_ACCUM_STEPS,
    'effective_batch_size': EFFECTIVE_BATCH_SIZE,
    'raw_global_only': True,
    'crop_used': False,
    'color_policy': 'RGB retained; random grayscale only as low-probability augmentation',
    'grayscale_probability': GRAYSCALE_PROBABILITY,
    'model_ema_decay': MODEL_EMA_DECAY,
    'gradient_checkpointing': USE_GRAD_CHECKPOINTING,
    'gpu_name': GPU_NAME,
    'gpu_vram_gib': GPU_VRAM_GIB,
    'online_train_augmentations': ['GPU horizontal flip', 'GPU small affine', 'GPU LAB-luminance CLAHE', 'GPU HSV', 'GPU random grayscale', 'GPU gamma', 'GPU blur', 'GPU JPEG', 'GPU downsample', 'GPU random erasing'],
    'augmentation_device': 'cuda',
    'cpu_pipeline': ['decode', 'EXIF transpose', 'RGB', 'letterbox', 'uint8 tensor'],
    'test_transform': ['EXIF transpose', 'RGB', 'letterbox', 'ImageNet normalization'],
    'flip_tta': USE_HORIZONTAL_FLIP_TTA,
    'external_training_images': False,
}
(OUTPUT_DIR / 'run_summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False))
archive_path = None
if CREATE_OUTPUT_ZIP:
    archive_path = shutil.make_archive(str(OUTPUT_DIR), 'zip', root_dir=OUTPUT_DIR)

display(submission.label.value_counts().reindex(CLASSES, fill_value=0).rename('predictions').to_frame())
display(submission.head())
print(json.dumps(summary, indent=2, ensure_ascii=False))
print('Submission:', OUTPUT_DIR / 'submission_convnext_base_student_384_amp_5fold_6epoch_a100.csv')
print('ZIP:', archive_path if archive_path else 'disabled untuk profil 3 jam')
